# IMPORTS

In [98]:
import os
import pandas as pd
import numpy as np
import re
import unicodedata
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle

# CONFIGURAÇÕES

In [99]:
BASE_FOLDER_TRAIN = "treino"

TRAIN_FILE = "ep2-train.csv"

preprocess_params = {
    "lowercase": False,
    "normalize_unicode": False,
    "remove_extra_whitespace": False,
    "remove_punct": False,
    
    "normalize_email": False,
    "normalize_url": False,
    "normalize_phone": False,
    "normalize_date": False,
    "normalize_time": False,
    "normalize_percent": False,
    "normalize_document": False,
    "normalize_code": False,
    "normalize_law": False,
}

In [100]:
# Configuração dos Pipelines e Grades de Hiperparâmetros para Grid Search
param_grids = {
    'Logistic Regression': {
        'pipeline': Pipeline([
            ('vectorizer', TfidfVectorizer(lowercase=preprocess_params['lowercase'])),
            ('model', LogisticRegression(max_iter=1000, random_state=42))
        ]),
        'params': {
            'vectorizer__max_features': [10000],
            'vectorizer__ngram_range': [(1, 2)],
            'model__C': [1.0],
            'model__solver': ['lbfgs'],
            'model__class_weight': ['balanced', None]
        }
    }
}


# ANÁLISE DE BALANCEAMENTO DOS DATASETS


In [101]:
def analisar_balanceamento(file_name):
    """Função para analisar o balanceamento de um dataset"""
    path = os.path.join(BASE_FOLDER_TRAIN, file_name)
    df = pd.read_csv(path, sep=";", encoding="latin1")
    
    print("="*60)
    print(f"ANÁLISE ESTATÍSTICA - {file_name}")
    print("="*60)
    
    # Informações básicas
    print(f"\n📊 INFORMAÇÕES GERAIS:")
    print(f"   • Total de linhas: {len(df):,}")
    print(f"   • Total de colunas: {len(df.columns)}")
    print(f"   • Colunas: {list(df.columns)}")
    
    # Verificar valores nulos
    print(f"\n🔍 VALORES NULOS:")
    print(f"   • Coluna 'req_text': {df['req_text'].isna().sum()}")
    print(f"   • Coluna 'profession': {df['profession'].isna().sum()}")
    
    # Distribuição das classes
    print(f"\n📈 DISTRIBUIÇÃO DAS CLASSES:")
    contagem_classes = df['profession'].value_counts()
    print(contagem_classes)
    
    print(f"\n📊 PORCENTAGEM POR CLASSE:")
    porcentagem_classes = df['profession'].value_counts(normalize=True) * 100
    for classe, perc in porcentagem_classes.items():
        count = contagem_classes[classe]
        print(f"   • {classe}: {count:,} ({perc:.2f}%)")
    
    # Verificar balanceamento
    print(f"\n⚖️ BALANCEAMENTO:")
    razao = contagem_classes.max() / contagem_classes.min()
    print(f"   • Razão maior/menor classe: {razao:.2f}x")
    if razao < 1.5:
        print(f"   • Status: ✅ Dataset bem balanceado")
    elif razao < 3:
        print(f"   • Status: ⚠️ Dataset moderadamente desbalanceado")
    else:
        print(f"   • Status: ❌ Dataset desbalanceado")
    
    print("\n" + "="*60)
    print()
    
    return df, contagem_classes


In [102]:
resultados_analise = {}

df, contagem = analisar_balanceamento(TRAIN_FILE)
resultados_analise[TRAIN_FILE] = {
    'dataframe': df,
    'contagem_classes': contagem
}


ANÁLISE ESTATÍSTICA - ep2-train.csv

📊 INFORMAÇÕES GERAIS:
   • Total de linhas: 43,678
   • Total de colunas: 2
   • Colunas: ['req_text', 'profession']

🔍 VALORES NULOS:
   • Coluna 'req_text': 0
   • Coluna 'profession': 0

📈 DISTRIBUIÇÃO DAS CLASSES:
profession
government    18782
academic      14593
private       10303
Name: count, dtype: int64

📊 PORCENTAGEM POR CLASSE:
   • government: 18,782 (43.00%)
   • academic: 14,593 (33.41%)
   • private: 10,303 (23.59%)

⚖️ BALANCEAMENTO:
   • Razão maior/menor classe: 1.82x
   • Status: ⚠️ Dataset moderadamente desbalanceado




# PRÉ-PROCESSAMENTO

In [103]:
def normalize_entities(text, params):
    """
    Normaliza entidades específicas no texto, substituindo por tokens especiais.
    Ordem de aplicação é importante para evitar conflitos!
    """
    if not isinstance(text, str):
        return ""
    
    # 1. EMAIL - Captura endereços de email
    if params.get("normalize_email", False):
        text = re.sub(
            r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
            '<EMAIL>',
            text
        )
    
    # 2. URL - Links HTTP/HTTPS e www (MUITO MAIS ROBUSTO)
    if params.get("normalize_url", False):
        # URLs com protocolo (http, https, ftp) - SEM \b para evitar problemas
        # Captura até encontrar espaço, aspas, ou pontuação de fim de frase
        text = re.sub(
            r'(?:https?|ftp)://[^\s\)"<>]+',
            '<URL>',
            text,
            flags=re.IGNORECASE
        )
        # URLs começando com www (incluindo www2, www3, etc)
        text = re.sub(
            r'www\d*\.[^\s\)"<>]+',
            '<URL>',
            text,
            flags=re.IGNORECASE
        )
        # Domínios com caminho explícito
        text = re.sub(
            r'[a-z0-9][-a-z0-9]*\.[a-z]{2,}(?:\.[a-z]{2,})?/[^\s\)"<>]+',
            '<URL>',
            text,
            flags=re.IGNORECASE
        )
    
    # 3. PHONE - Telefones brasileiros
    if params.get("normalize_phone", False):
        # Formato: (85) 9 8645.1524, (85) 98645-1524, 85 98645-1524, etc
        text = re.sub(
            r'\(?\d{2}\)?\s?\d{4,5}[-.\s]?\d{4}',
            '<PHONE>',
            text
        )
    
    # 4. DATE - Várias formatações de data
    if params.get("normalize_date", False):
        # DD/MM/YYYY, DD-MM-YYYY, DD.MM.YYYY
        text = re.sub(
            r'\b\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}\b',
            '<DATE>',
            text
        )
        # YYYY/MM/DD, YYYY-MM-DD
        text = re.sub(
            r'\b\d{4}[/.-]\d{1,2}[/.-]\d{1,2}\b',
            '<DATE>',
            text
        )
    
    # 5. TIME - Horários (CORRIGIDO)
    if params.get("normalize_time", False):
        # Formato HH:MM ou HH:MM:SS
        text = re.sub(
            r'\b\d{1,2}:\d{2}(?::\d{2})?\b',
            '<TIME>',
            text
        )
        # Formato HHhMM (ex: 14h30)
        text = re.sub(
            r'\b\d{1,2}h\d{2}\b',
            '<TIME>',
            text
        )
    
    # 6. PERCENT - Porcentagens
    if params.get("normalize_percent", False):
        text = re.sub(
            r'\b\d+(?:[.,]\d+)?%',
            '<PERCENT>',
            text
        )
    
    # 7. DOCUMENT - CPF e CNPJ
    if params.get("normalize_document", False):
        # CPF: 123.456.789-00
        text = re.sub(
            r'\b\d{3}\.\d{3}\.\d{3}-\d{2}\b',
            '<DOCUMENT>',
            text
        )
        # CNPJ: 12.345.678/0001-00
        text = re.sub(
            r'\b\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}\b',
            '<DOCUMENT>',
            text
        )
    
    # 8. CODE - Códigos de rastreamento e processos
    if params.get("normalize_code", False):
        # Códigos de rastreamento (ex: RU101805325NL, PG326875631BR)
        text = re.sub(
            r'\b[A-Z]{2}\d{9,}[A-Z]{2}\b',
            '<CODE>',
            text
        )
        # Códigos de processo (ex: AC-2000-08012-000640, DAE 07.16.17365.94815-1)
        text = re.sub(
            r'\b[A-Z]{2,}-?\d{4}-\d{5}-\d{5,6}(?:-\d)?\b',
            '<CODE>',
            text
        )
        # DAE com pontos
        text = re.sub(
            r'\b\d{2}\.\d{2}\.\d{5}\.\d{5}-\d\b',
            '<CODE>',
            text
        )
    
    # 9. LAW - Referências legais
    if params.get("normalize_law", False):
        text = re.sub(
            r'\b(?:Lei|Portaria|Decreto|Resolução|Edital)\s+n[ºo°]?\s*\d+(?:/\d{4})?\b',
            '<LAW>',
            text,
            flags=re.IGNORECASE
        )
    
    return text


In [104]:
def preprocess_operations(text, params):
    """
    Aplica operações de pré-processamento no texto.
    ORDEM IMPORTANTE: Normalização de entidades ANTES de lowercase e remoção de pontuação!
    """
    if not isinstance(text, str):
        return ""
    
    # PASSO 1: Normalização de entidades (PRIMEIRO - antes de alterar case ou pontuação)
    text = normalize_entities(text, params)
    
    # PASSO 2: Normalização Unicode
    if params.get("normalize_unicode", True):
        text = unicodedata.normalize("NFKC", text)
    
    # PASSO 3: Lowercase
    if params.get("lowercase", True):
        text = text.lower()
    
    # PASSO 4: Remover pontuação (mas preservar tokens especiais <...>)
    if params.get("remove_punct", True):
        # Proteger tokens especiais temporariamente
        text = re.sub(r'<(\w+)>', r'SPECIALTOKEN\1SPECIALTOKEN', text)
        # Remover pontuação
        text = re.sub(r"[^\w\s]", " ", text)
        # Restaurar tokens especiais
        text = re.sub(r'SPECIALTOKEN(\w+)SPECIALTOKEN', r'<\1>', text)
    
    # PASSO 5: Remover espaços extras
    if params.get("remove_extra_whitespace", True):
        text = re.sub(r"\s+", " ", text).strip()
    
    return text

def preprocess_data(path, output_path, params):
    """
    Processa dados, salva no CSV e retorna dados preparados para treinamento.
    """
    col_text, col_label = "req_text", "profession"
    
    # 1. Carregar dados originais
    if not os.path.exists(path):
        print(f"Aviso: {path} não encontrado.")
        return None
    
    df = pd.read_csv(path, sep=";", encoding="latin1")
    df = df[[col_text, col_label]].dropna()
    
    # 2. Aplicar pré-processamento
    df['req_text'] = df[col_text].apply(lambda x: preprocess_operations(x, params))
    
    # 3. Manter apenas texto processado e profissão
    df_processed = df[['req_text', col_label]].copy()
    
    # 4. Salvar no CSV
    df_processed.to_csv(output_path, sep=";", encoding="utf-8", index=False)
    
    # 5. Preparar para treinamento
    df_processed = shuffle(df_processed, random_state=10).reset_index(drop=True)
    
    le = LabelEncoder()
    y = le.fit_transform(df_processed[col_label])
    X = df_processed['req_text'].values
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.15, stratify=y, random_state=10
    )
    
    return X_train, X_test, y_train, y_test

In [105]:
# PROCESSAR DADOS, SALVAR NO CSV E PREPARAR PARA TREINAMENTO

print("="*100)
print("PROCESSANDO DADOS")
print("="*100)

input_file = os.path.join(BASE_FOLDER_TRAIN, TRAIN_FILE)
output_file = os.path.join(BASE_FOLDER_TRAIN, "ep2-train-preprocessed.csv")

print(f"\n📂 Processando: {TRAIN_FILE}")

# Processar, salvar no CSV e preparar para treinamento (tudo em uma função)
result = preprocess_data(input_file, output_file, preprocess_params)

if result is not None:
    X_train, X_test, y_train, y_test = result
    dataset = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }
    print(f"   ✅ Pré-processamento concluído e salvo em: {output_file}")
    print(f"   ✅ Dados preparados para treinamento!")
    print(f"   • Treino: {len(X_train)} textos | Teste: {len(X_test)} textos")
else:
    print(f"   ❌ Erro ao processar dados")
    dataset = {}

PROCESSANDO DADOS

📂 Processando: ep2-train.csv
   ✅ Pré-processamento concluído e salvo em: treino/ep2-train-preprocessed.csv
   ✅ Dados preparados para treinamento!
   • Treino: 37126 textos | Teste: 6552 textos


# TREINAMENTO

Classificação de textos entre autores **acadêmicos**, **privados** e **governamentais**.

In [106]:
X_train = dataset["X_train"]
X_test = dataset["X_test"]
y_train = dataset["y_train"]
y_test = dataset["y_test"]

print(f"Dados carregados")
print(f"   Treino: {len(X_train)} textos | Teste: {len(X_test)} textos")


Dados carregados
   Treino: 37126 textos | Teste: 6552 textos


In [107]:
print("="*80)
print("GRID SEARCH COM PIPELINE")
print("="*80)
print("Otimizando TF-IDF + Modelos simultaneamente...\n")

# Armazenar melhores pipelines
best_models = {}
cv_results = {}

for name, config in param_grids.items():
    print(f"[{name}] Executando Grid Search...")
    print(f"   Testando {len(config['params']['vectorizer__max_features']) * len(config['params']['vectorizer__ngram_range'])} combinações de TF-IDF...")
    
    # Grid Search com 10-fold CV
    grid_search = GridSearchCV(
        config['pipeline'],
        config['params'],
        cv=10,
        scoring='accuracy',
        n_jobs=-1,
        verbose=0
    )
    
    grid_search.fit(X_train, y_train)
    
    # Armazenar resultados
    best_models[name] = grid_search.best_estimator_
    
    # Separar parâmetros de TF-IDF e modelo
    vectorizer_params = {k.replace('vectorizer__', ''): v 
                        for k, v in grid_search.best_params_.items() 
                        if k.startswith('vectorizer__')}
    model_params = {k.replace('model__', ''): v 
                   for k, v in grid_search.best_params_.items() 
                   if k.startswith('model__')}
    
    cv_results[name] = {
        'best_params': grid_search.best_params_,
        'vectorizer_params': vectorizer_params,
        'model_params': model_params,
        'best_score': grid_search.best_score_,
        'mean': grid_search.best_score_,
        'std': grid_search.cv_results_['std_test_score'][grid_search.best_index_]
    }
    
    print(f"  ✓ Melhores params TF-IDF: {vectorizer_params}")
    print(f"  ✓ Melhores params Modelo: {model_params}")
    print(f"  ✓ Acuracia (CV): {grid_search.best_score_:.4f}\n")

print("="*80)


GRID SEARCH COM PIPELINE
Otimizando TF-IDF + Modelos simultaneamente...

[Logistic Regression] Executando Grid Search...
   Testando 1 combinações de TF-IDF...


  ✓ Melhores params TF-IDF: {'max_features': 10000, 'ngram_range': (1, 2)}
  ✓ Melhores params Modelo: {'C': 1.0, 'class_weight': None, 'solver': 'lbfgs'}
  ✓ Acuracia (CV): 0.6574

